# P1.4 / P1.2 on a GPU — tests, paging benchmark, profiling

Runs this repo's serving code on a Colab GPU: the full test suite (CUDA variants included),
the paged-vs-row benchmark, and the profiler that separates the three hypotheses in
`docs/benchmarks/paging.md`.

**Runtime → Change runtime type → GPU** before running. Weights are random — speed and
memory do not depend on them.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!git clone -q https://github.com/shuzhancnjx/gpt2-from-scratch.git
%cd gpt2-from-scratch
!pip install -q -e ".[dev]"

## 1 · Tests — CPU and CUDA
The `device` fixture now runs the CUDA variants that were skipped on the Mac.

In [ ]:
!python -m pytest tests/gpt2 -q -k "paged or block_allocator or serving" 

## 2 · Benchmark — same KV memory, row vs paged (fp16, peak memory included)

In [ ]:
!python scripts/bench_paging.py --device cuda --out results/bench_paging_cuda.json
!python scripts/bench_paging.py --device cuda --paged-seqs 32 --out results/bench_paging_cuda_seq32.json

## 3 · Profile — where does the time go?
Part A: prefill share and ms per decode step as `max_seqs` grows.
Part B: top CUDA operators and a Chrome trace of one paged run.

In [ ]:
!python scripts/profile_serving.py --device cuda --trace-dir results/traces

In [ ]:
# download the trace, then open it at https://ui.perfetto.dev
import glob
from google.colab import files
for f in glob.glob('results/traces/*.json') + glob.glob('results/bench_paging_cuda*.json'):
    files.download(f)

## 4 · Your analysis (judged)

Answer in `docs/benchmarks/paging.md`, using the numbers above:

1. **Hypothesis 3** — what share of wall time is prefill? Does it explain the missing gain?
2. **Hypotheses 1 and 2** — how does ms per decode step grow as `max_seqs` goes 4 → 32?
   Is that growth proportional to `max_seqs × W` (gather + full-width attention) or to
   the tokens actually resident?
3. In Part B, which operators dominate a decode step? Where does `aten::index` (the
   gather) rank against the matmuls and attention?
4. Predict, before P1.5: if attention read only real blocks, which number above would
   change, and by roughly how much?